# Deriving eigengenes and phenotypes correlation matrix from SpeakEasy2 output

In [2]:
source("/media/psylab-6028/DATA1/Eden/CoExpression_ReProduction/old_scripts/ClusteringBuilding.R")

Loading required package: dynamicTreeCut

Loading required package: fastcluster




Attaching package: ‘fastcluster’


The following object is masked from ‘package:stats’:

    hclust





Attaching package: ‘WGCNA’


The following object is masked from ‘package:stats’:

    cor




Allowing multi-threading with up to 32 threads.
Allowing multi-threading with up to 32 threads.


In [ ]:
max_genes_per_tissue <- 5000L
sd_quantile <- 0.00
tissue_names <- c("AC", "PCGBA23", "MFBA9BA46")
rosmap_files <- c(
    "/media/psylab-6028/DATA1/Eden/CoExpression_ReProduction/nbs/ROSMAP_fixed_AC.csv",
    "/media/psylab-6028/DATA1/Eden/CoExpression_ReProduction/nbs/ROSMAP_fixed_PCGBA23.csv",
    "/media/psylab-6028/DATA1/Eden/CoExpression_ReProduction/nbs/ROSMAP_fixed_MFBA9BA46.csv"
)
# Read the tsv file of the clusters table and details
clusters_table <- read.csv(
    "/media/psylab-6028/DATA1/Eden/speakeasyR/SLabPL/speakeasy_clusters/speakeasy_clusters_table_3.csv",
    sep = ",",
    stringsAsFactors = FALSE
)
out_prefix <- "/media/psylab-6028/DATA1/Eden/CoExpression_ReProduction/nbs/speakeasy2_eigengenes_3_"

In [8]:
# Build donor mats + donors intersection
tmp <- build_donor_mats_and_common(tissue_names, rosmap_files,
                                    sd_quantile, max_genes_per_tissue)
donor_mats <- tmp$donor_mats

# Gene metadata from your clusters table
gene_meta <- build_gene_metadata(clusters_table)


In [10]:
res_MEs <- compute_all_module_eigengenes_common(
    tissue_names = tissue_names,
    tissue_expr_file_names = rosmap_files,
    clusters_table = clusters_table,   
    sd_quantile = sd_quantile,
    max_genes_per_tissue = max_genes_per_tissue,
    min_genes_for_ME = 1L,            
    center_scale = TRUE,
    export_prefix = paste0(out_prefix, "_ME_commonDonors")
)

In [14]:
me_files <- c(
  AC          = "/media/psylab-6028/DATA1/Eden/CoExpression_ReProduction/nbs/speakeasy2_eigengenes_3__ME_commonDonors_AC_per_tissue.tsv",
  PCGBA23     = "/media/psylab-6028/DATA1/Eden/CoExpression_ReProduction/nbs/speakeasy2_eigengenes_3__ME_commonDonors_PCGBA23_per_tissue.tsv",
  MFBA9BA46   = "/media/psylab-6028/DATA1/Eden/CoExpression_ReProduction/nbs/speakeasy2_eigengenes_3__ME_commonDonors_MFBA9BA46_per_tissue.tsv",
  CROSS       = "/media/psylab-6028/DATA1/Eden/CoExpression_ReProduction/nbs/speakeasy2_eigengenes_3__ME_commonDonors_cross_tissue.tsv"   # optional, can remove if not used
)

pheno_file <- "/media/psylab-6028/DATA1/Eden/CoExpression_ReProduction/nbs/dataset_1593_cross-sectional_08-13-2025.xlsx"  # or your local path
pheno_sheet <- 1   

out_prefix <- "/media/psylab-6028/DATA1/Eden/CoExpression_ReProduction/nbs/rosmap_SE2_3_ME_pheno"


In [15]:
suppressPackageStartupMessages({
  library(readr); library(readxl); library(dplyr); library(tidyr); library(purrr)
  library(stringr); library(broom)
})

donor_to_projid <- function(x) {
  x <- gsub("^ROSMAP-", "", x)
  x <- sub("^0+", "", x)           
  x[x == ""] <- NA_character_      
  as.integer(x)
}

read_me_table <- function(path, tissue_label = NULL) {
  tb <- suppressMessages(readr::read_tsv(path, col_types = cols(.default = col_guess())))
  if (!("donor" %in% names(tb))) {
    # Assume first column is donor if not explicitly named
    names(tb)[1] <- "donor"
  }
  stopifnot("donor" %in% names(tb))
  tb <- tb %>% mutate(projid = donor_to_projid(donor)) %>% select(projid, everything())
  mod_cols <- setdiff(names(tb), c("projid","donor"))
  ord <- order(as.integer(sub("^M","", mod_cols)), na.last = TRUE)
  tb <- tb %>% select(projid, all_of(c("donor", mod_cols[ord])))
  if (!is.null(tissue_label)) attr(tb, "tissue") <- tissue_label
  tb
}

# Read phenotype (Excel) and keep projid as integer
read_pheno <- function(xlsx_path, sheet = 1) {
  ph <- readxl::read_excel(xlsx_path, sheet = sheet)
  # Try to find a projid-like column
  cn <- names(ph)
  pid_col <- cn[grepl("^projid$", cn, ignore.case = TRUE)][1]
  if (is.na(pid_col)) stop("Could not find a 'projid' column in the phenotype sheet.")
  ph <- ph %>%
    rename(projid = !!sym(pid_col)) %>%
    mutate(projid = as.integer(projid))
  ph
}

correlate_me_vs_pheno <- function(me_df, pheno_df,
                                  phenotypes = NULL,
                                  method = c("pearson","spearman"),
                                  min_n = 10,
                                  adjust = "BH",
                                  tissue_label = NULL) {
  method <- match.arg(method)

  dat <- inner_join(me_df, pheno_df, by = "projid")
  if (!nrow(dat)) return(tibble())

  me_cols <- grep("^M\\d+$", names(dat), value = TRUE)
  if (!length(me_cols)) stop("No module columns (M<number>) found in ME table.")

  if (is.null(phenotypes)) {
    num_cols <- names(dat)[vapply(dat, is.numeric, TRUE)]
    phenotypes <- setdiff(num_cols, c("projid", me_cols))
  } else {
    phenotypes <- intersect(phenotypes, names(dat))
  }
  if (!length(phenotypes)) stop("No numeric phenotype columns found to correlate.")

  long_me <- dat %>% select(projid, all_of(me_cols)) %>% pivot_longer(-projid, names_to = "module", values_to = "ME")
  long_ph <- dat %>% select(projid, all_of(phenotypes)) %>% pivot_longer(-projid, names_to = "phenotype", values_to = "Y")

  cj <- inner_join(long_me, long_ph, by = "projid")

  res <- cj %>%
    group_by(module, phenotype) %>%
    summarize(
      n = sum(is.finite(ME) & is.finite(Y)),
      cor = suppressWarnings(ifelse(n >= min_n, cor(ME, Y, use = "pairwise.complete.obs", method = method), NA_real_)),
      .groups = "drop"
    ) %>%
    mutate(
      t_stat = cor * sqrt(pmax(n - 2, 0) / pmax(1 - cor^2, .Machine$double.eps)),
      p = ifelse(n >= 3, 2 * pt(-abs(t_stat), df = pmax(n - 2, 1)), NA_real_)
    ) %>%
    group_by(phenotype) %>%          # adjust within each phenotype (common choice)
    mutate(p_adj = p.adjust(p, method = adjust)) %>%
    ungroup() %>%
    arrange(p_adj, phenotype, module)

  if (!is.null(tissue_label)) res$tissue <- tissue_label
  res %>% select(tissue, module, phenotype, n, cor, p, p_adj, t_stat)
}


In [16]:
# Read phenotypes
pheno <- read_pheno(pheno_file, sheet = pheno_sheet)

# Read each ME table and correlate
results_list <- list()

# Per-tissue
for (nm in setdiff(names(me_files), "CROSS")) {
  me_tb <- read_me_table(me_files[[nm]], tissue_label = nm)
  results_list[[nm]] <- correlate_me_vs_pheno(
    me_df = me_tb, pheno_df = pheno,
    phenotypes = NULL,         # or c("age_death","amyloid","braak","cdr", ...)
    method = "pearson",
    min_n = 20,
    adjust = "BH",
    tissue_label = nm
  )
}

# Cross-tissue (optional)
if ("CROSS" %in% names(me_files)) {
  me_tb <- read_me_table(me_files[["CROSS"]], tissue_label = "CROSS")
  results_list[["CROSS"]] <- correlate_me_vs_pheno(
    me_df = me_tb, pheno_df = pheno,
    phenotypes = NULL,
    method = "pearson",
    min_n = 20,
    adjust = "BH",
    tissue_label = "CROSS"
  )
}

# Bind all results and write to disk
all_res <- bind_rows(results_list)
readr::write_tsv(all_res, paste0(out_prefix, "_ME_vs_pheno_correlations.tsv"))

# Quick peek: top hits
print(dplyr::slice_min(all_res, p_adj, n = 20))


Warning message in inner_join(long_me, long_ph, by = "projid"):
“Detected an unexpected many-to-many relationship between `x` and `y`.
ℹ Row 1 of `x` matches multiple rows in `y`.
ℹ Row 1 of `y` matches multiple rows in `x`.
ℹ If a many-to-many relationship is expected, set `relationship =
  "many-to-many"` to silence this warning.”
Warning message in inner_join(long_me, long_ph, by = "projid"):
“Detected an unexpected many-to-many relationship between `x` and `y`.
ℹ Row 1 of `x` matches multiple rows in `y`.
ℹ Row 1 of `y` matches multiple rows in `x`.
ℹ If a many-to-many relationship is expected, set `relationship =
  "many-to-many"` to silence this warning.”
Warning message in inner_join(long_me, long_ph, by = "projid"):
“Detected an unexpected many-to-many relationship between `x` and `y`.
ℹ Row 1 of `x` matches multiple rows in `y`.
ℹ Row 1 of `y` matches multiple rows in `x`.
ℹ If a many-to-many relationship is expected, set `relationship =
  "many-to-many"` to silence this warni

# A tibble: 20 × 8
   tissue    module phenotype             n    cor        p    p_adj t_stat
   <chr>     <chr>  <chr>             <int>  <dbl>    <dbl>    <dbl>  <dbl>
 1 CROSS     M224   msex                457  0.468 2.83e-26 3.79e-24  11.3 
 2 AC        M224   msex                457  0.355 5.31e-15 6.85e-13   8.09
 3 PCGBA23   M49    cogdx               457  0.297 9.72e-11 1.44e- 8   6.63
 4 CROSS     M49    cogdx               457 -0.292 1.83e-10 2.45e- 8  -6.52
 5 MFBA9BA46 M224   msex                457  0.290 2.58e-10 2.68e- 8   6.47
 6 PCGBA23   M203   cogdx               457 -0.282 8.54e-10 6.32e- 8  -6.27
 7 MFBA9BA46 M203   cogdx               457  0.281 9.53e-10 9.91e- 8   6.25
 8 AC        M17    age_death           457 -0.282 8.51e-10 1.10e- 7  -6.27
 9 AC        M202   c_score             457  0.281 9.52e-10 1.23e- 7   6.25
10 AC        M202   ceradsc             457 -0.281 9.52e-10 1.23e- 7  -6.25
11 AC        M202   gpath               457  0.281 1.02e- 9 1.32e- 7 